# WikiANN — Albanian (`sq`) quick look

Tiny exploration of the dataset we baseline against: split sizes, label distribution, sentence length. Sanity-check before running `run_baselines.py`.

In [ ]:
from collections import Counter

import pandas as pd
from datasets import load_dataset

ds = load_dataset("unimelb-nlp/wikiann", "sq")
label_names = ds["train"].features["ner_tags"].feature.names
label_names

In [2]:
pd.DataFrame({split: [len(ds[split])] for split in ds}, index=["sentences"])

,validation,test,train
sentences,1000,1000,5000


In [3]:
def entity_counts(split):
    c = Counter()
    for row in ds[split]:
        for tag_id in row["ner_tags"]:
            name = label_names[tag_id]
            if name.startswith("B-"):
                c[name[2:]] += 1
    return c

pd.DataFrame({split: entity_counts(split) for split in ds}).fillna(0).astype(int)

,validation,test,train
LOC,368,338,1822
ORG,326,382,1727
PER,363,354,1802


In [4]:
lens = [len(r["tokens"]) for r in ds["test"]]
pd.Series(lens).describe()

count    1000.00000
mean        7.52400
std         5.39152
min         3.00000
25%         5.00000
50%         6.00000
75%         8.00000
max        67.00000
dtype: float64

In [ ]:
# A peek at three test sentences with their gold tags.
for row in ds["test"].select(range(3)):
    tags = [label_names[i] for i in row["ner_tags"]]
    for tok, tag in zip(row["tokens"], tags, strict=True):
        marker = "" if tag == "O" else f"   <- {tag}"
        print(f"  {tok:25s} {marker}")
    print("---")